# Prepare Real TLC Replay Assets

Prepare the real January/February 2023 TLC files for the Ray runtime. The local run uses the first 48 replay ticks so the notebook stays quick.

Imports and constants used by this notebook.

In [1]:
from __future__ import annotations
import json
import os
from pathlib import Path
from typing import Any
import pandas as pd

PICKUP_COL = "lpep_pickup_datetime"
ZONE_COL = "PULocationID"


Define local JSON helpers so this notebook can run without importing a Python script.

In [2]:
def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    if isinstance(value, tuple):
        return [json_safe(item) for item in value]
    if hasattr(value, "item"):
        return value.item()
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return value


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True), encoding="utf-8")


def load_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


Define the `require_columns` function.

In [3]:
def require_columns(frame: pd.DataFrame, path: Path) -> None:
    missing = [col for col in (PICKUP_COL, ZONE_COL) if col not in frame.columns]
    if missing:
        raise ValueError(f"{path} is missing required column(s): {missing}")


Define the `dominant_month` function.

In [4]:
def dominant_month(frame: pd.DataFrame, label: str) -> tuple[pd.Period, dict[str, int]]:
    month_counts = (
        pd.to_datetime(frame[PICKUP_COL], errors="coerce")
        .dropna()
        .dt.to_period("M")
        .value_counts()
        .sort_values(ascending=False)
    )
    if month_counts.empty:
        raise ValueError(f"{label} data has no valid pickup datetimes")
    month = month_counts.index[0]
    return month, {str(key): int(value) for key, value in month_counts.items()}


Define the `validate_adjacent_months` function.

In [5]:
def validate_adjacent_months(reference: pd.DataFrame, replay: pd.DataFrame) -> tuple[pd.Period, pd.Period, dict[str, int], dict[str, int]]:
    ref_month, ref_counts = dominant_month(reference, "reference")
    replay_month, replay_counts = dominant_month(replay, "replay")
    if ref_month.year != replay_month.year:
        raise ValueError("reference and replay months must be in the same year")
    if replay_month.month != ref_month.month + 1:
        raise ValueError(
            f"replay month must immediately follow reference month; got {ref_month} -> {replay_month}"
        )
    return ref_month, replay_month, ref_counts, replay_counts


Define the `add_tick_columns` function.

In [6]:
def add_tick_columns(frame: pd.DataFrame, tick_minutes: int) -> pd.DataFrame:
    out = frame[[PICKUP_COL, ZONE_COL]].copy()
    out[PICKUP_COL] = pd.to_datetime(out[PICKUP_COL], errors="coerce")
    out[ZONE_COL] = pd.to_numeric(out[ZONE_COL], errors="coerce")
    out = out.dropna(subset=[PICKUP_COL, ZONE_COL]).copy()
    out["zone_id"] = out[ZONE_COL].astype(int)
    out["tick_start"] = out[PICKUP_COL].dt.floor(f"{tick_minutes}min")
    out["hour_of_day"] = out["tick_start"].dt.hour.astype(int)
    out["day_of_week"] = out["tick_start"].dt.dayofweek.astype(int)
    return out


Define the `filter_to_month` function.

In [7]:
def filter_to_month(frame: pd.DataFrame, month: pd.Period) -> pd.DataFrame:
    return frame[frame["tick_start"].dt.to_period("M") == month].copy()


Define the `select_active_zones` function.

In [8]:
def select_active_zones(reference: pd.DataFrame, n_zones: int) -> list[int]:
    zone_counts = (
        reference.groupby("zone_id")
        .size()
        .rename("pickup_count")
        .reset_index()
        .sort_values(["pickup_count", "zone_id"], ascending=[False, True])
    )
    if zone_counts.empty:
        raise ValueError("reference data has no pickup zones")
    return [int(zone_id) for zone_id in zone_counts.head(n_zones)["zone_id"]]


Define the `aggregate_counts` function.

In [9]:
def aggregate_counts(frame: pd.DataFrame) -> pd.DataFrame:
    return (
        frame.groupby(["zone_id", "tick_start", "hour_of_day", "day_of_week"])
        .size()
        .rename("demand_count")
        .reset_index()
    )


Define the `build_baseline` function.

In [10]:
def build_baseline(reference_counts: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, float]:
    baseline = (
        reference_counts.groupby(["zone_id", "hour_of_day", "day_of_week"])["demand_count"]
        .mean()
        .rename("baseline_count")
        .reset_index()
    )
    zone_defaults = reference_counts.groupby("zone_id")["demand_count"].mean()
    global_default = float(reference_counts["demand_count"].mean())
    return baseline, zone_defaults, global_default


Define the `build_replay_table` function.

In [11]:
def build_replay_table(
    replay: pd.DataFrame,
    active_zones: list[int],
    baseline: pd.DataFrame,
    zone_defaults: pd.Series,
    global_default: float,
    tick_minutes: int,
) -> pd.DataFrame:
    replay_counts = aggregate_counts(replay[replay["zone_id"].isin(active_zones)])
    if replay_counts.empty:
        raise ValueError("replay data has no pickups for the selected active zones")

    tick_starts = pd.date_range(
        start=replay_counts["tick_start"].min(),
        end=replay_counts["tick_start"].max(),
        freq=f"{tick_minutes}min",
    )
    grid = pd.MultiIndex.from_product(
        [active_zones, tick_starts], names=["zone_id", "tick_start"]
    ).to_frame(index=False)
    grid["hour_of_day"] = grid["tick_start"].dt.hour.astype(int)
    grid["day_of_week"] = grid["tick_start"].dt.dayofweek.astype(int)

    replay_table = grid.merge(
        replay_counts[["zone_id", "tick_start", "demand_count"]],
        on=["zone_id", "tick_start"],
        how="left",
    )
    replay_table["demand_count"] = replay_table["demand_count"].fillna(0).astype(int)
    replay_table = replay_table.merge(
        baseline, on=["zone_id", "hour_of_day", "day_of_week"], how="left"
    )
    replay_table["baseline_count"] = replay_table["baseline_count"].fillna(
        replay_table["zone_id"].map(zone_defaults)
    )
    replay_table["baseline_count"] = replay_table["baseline_count"].fillna(global_default)
    tick_ids = {tick: idx for idx, tick in enumerate(sorted(tick_starts))}
    replay_table["tick_id"] = replay_table["tick_start"].map(tick_ids).astype(int)
    return replay_table.sort_values(["tick_id", "zone_id"]).reset_index(drop=True)


Define the `cross_check_replay` function.

In [12]:
def cross_check_replay(
    raw_replay: pd.DataFrame,
    replay_table: pd.DataFrame,
    active_zones: list[int],
    sample_ticks: int,
    tick_minutes: int,
) -> dict[str, object]:
    tick_values = sorted(replay_table["tick_start"].unique())
    selected_ticks = tick_values[: min(sample_ticks, len(tick_values))]
    sample_start = pd.Timestamp(selected_ticks[0])
    sample_end = pd.Timestamp(selected_ticks[-1]) + pd.Timedelta(minutes=tick_minutes)

    direct = raw_replay[
        raw_replay["zone_id"].isin(active_zones)
        & (raw_replay["tick_start"] >= sample_start)
        & (raw_replay["tick_start"] < sample_end)
    ]
    direct_total = int(len(direct))
    prepared_total = int(
        replay_table[replay_table["tick_start"].isin(selected_ticks)]["demand_count"].sum()
    )
    return {
        "sample_start": sample_start.isoformat(),
        "sample_end_exclusive": sample_end.isoformat(),
        "sample_ticks": len(selected_ticks),
        "direct_total": direct_total,
        "prepared_total": prepared_total,
        "passed": direct_total == prepared_total,
    }


Define the `prepare_assets` function.

In [13]:
def prepare_assets(
    reference_parquet: Path,
    replay_parquet: Path,
    output_dir: Path,
    n_zones: int,
    tick_minutes: int,
    seed: int,
    max_ticks: int | None,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    reference_raw = pd.read_parquet(reference_parquet)
    replay_raw = pd.read_parquet(replay_parquet)
    require_columns(reference_raw, reference_parquet)
    require_columns(replay_raw, replay_parquet)
    ref_month, replay_month, ref_month_counts, replay_month_counts = validate_adjacent_months(
        reference_raw, replay_raw
    )

    reference = filter_to_month(add_tick_columns(reference_raw, tick_minutes), ref_month)
    replay = filter_to_month(add_tick_columns(replay_raw, tick_minutes), replay_month)
    active_zones = select_active_zones(reference, n_zones=n_zones)

    reference_counts = aggregate_counts(reference[reference["zone_id"].isin(active_zones)])
    baseline, zone_defaults, global_default = build_baseline(reference_counts)
    replay_table = build_replay_table(
        replay=replay,
        active_zones=active_zones,
        baseline=baseline,
        zone_defaults=zone_defaults,
        global_default=global_default,
        tick_minutes=tick_minutes,
    )
    if max_ticks is not None:
        keep_tick_ids = sorted(replay_table["tick_id"].unique())[:max_ticks]
        replay_table = replay_table[replay_table["tick_id"].isin(keep_tick_ids)].copy()
    check = cross_check_replay(
        raw_replay=replay,
        replay_table=replay_table,
        active_zones=active_zones,
        sample_ticks=8,
        tick_minutes=tick_minutes,
    )
    if not check["passed"]:
        raise ValueError(f"prepared replay cross-check failed: {check}")

    baseline.to_parquet(output_dir / "baseline.parquet", index=False)
    replay_table.to_parquet(output_dir / "replay_table.parquet", index=False)
    write_json(output_dir / "active_zones.json", active_zones)
    write_json(output_dir / "cross_check.json", check)
    write_json(
        output_dir / "prepare_config.json",
        {
            "reference_parquet": str(reference_parquet),
            "replay_parquet": str(replay_parquet),
            "reference_month": str(ref_month),
            "replay_month": str(replay_month),
            "reference_month_counts": ref_month_counts,
            "replay_month_counts": replay_month_counts,
            "n_zones": len(active_zones),
            "tick_minutes": tick_minutes,
            "seed": seed,
            "max_ticks": max_ticks,
            "first_use_fallback_decision": "OK",
        },
    )

    print(f"Prepared {len(active_zones)} active zones and {replay_table['tick_id'].nunique()} ticks")
    print(f"Cross-check passed: {check}")


Run the preparation step on real TLC data.

In [14]:
data_dir = Path(os.getenv("CAPSTONE_DATA_DIR", "data"))
prepared_dir = Path(os.getenv("CAPSTONE_PREPARED_DIR", "prepared"))
prepare_assets(
    reference_parquet=data_dir / "green_tripdata_2023-01.parquet",
    replay_parquet=data_dir / "green_tripdata_2023-02.parquet",
    output_dir=prepared_dir,
    n_zones=6,
    tick_minutes=15,
    seed=22971,
    max_ticks=48,
)


Prepared 6 active zones and 48 ticks
Cross-check passed: {'sample_start': '2023-02-01T00:00:00', 'sample_end_exclusive': '2023-02-01T02:00:00', 'sample_ticks': 8, 'direct_total': 17, 'prepared_total': 17, 'passed': True}


Inspect preparation outputs.

In [15]:
cross_check = load_json(prepared_dir / "cross_check.json")
prepare_config = load_json(prepared_dir / "prepare_config.json")
replay_table = pd.read_parquet(prepared_dir / "replay_table.parquet")
active_zones = load_json(prepared_dir / "active_zones.json")

prepare_summary_path = prepared_dir / "prepare_summary.md"
prepare_summary = [
    "# Preparation Evidence",
    "",
    f"- Active zones: {len(active_zones)}",
    f"- Replay ticks: {prepare_config['max_ticks']}",
    f"- Tick length minutes: {prepare_config['tick_minutes']}",
    f"- Replay table rows: {len(replay_table)}",
    f"- Cross-check sample ticks: {cross_check['sample_ticks']}",
    f"- Raw replay rows in sample: {cross_check['direct_total']}",
    f"- Prepared demand total in sample: {cross_check['prepared_total']}",
    f"- Cross-check passed: {cross_check['passed']}",
]
prepare_summary_path.write_text("\n".join(prepare_summary) + "\n", encoding="utf-8")

print("Preparation evidence")
print("--------------------")
print(cross_check)
print(prepare_config)
print(f"prepare summary saved to: {prepare_summary_path}")
display(replay_table.head(12))


{'direct_total': 17, 'passed': True, 'prepared_total': 17, 'sample_end_exclusive': '2023-02-01T02:00:00', 'sample_start': '2023-02-01T00:00:00', 'sample_ticks': 8}
{'first_use_fallback_decision': 'OK', 'max_ticks': 48, 'n_zones': 6, 'reference_month': '2023-01', 'reference_month_counts': {'2009-01': 1, '2022-12': 2, '2023-01': 68207, '2023-02': 1}, 'reference_parquet': 'data\\green_tripdata_2023-01.parquet', 'replay_month': '2023-02', 'replay_month_counts': {'2008-12': 1, '2023-01': 24, '2023-02': 64783, '2023-03': 1}, 'replay_parquet': 'data\\green_tripdata_2023-02.parquet', 'seed': 22971, 'tick_minutes': 15}


,zone_id,tick_start,hour_of_day,day_of_week,demand_count,baseline_count,tick_id
0,41,2023-02-01 00:00:00,0,2,0,1.333333,0
1,74,2023-02-01 00:00:00,0,2,0,1.000000,0
2,75,2023-02-01 00:00:00,0,2,0,1.333333,0
3,82,2023-02-01 00:00:00,0,2,1,1.250000,0
4,95,2023-02-01 00:00:00,0,2,2,2.000000,0
5,166,2023-02-01 00:00:00,0,2,0,1.000000,0
6,41,2023-02-01 00:15:00,0,2,0,1.333333,1
7,74,2023-02-01 00:15:00,0,2,0,1.000000,1
8,75,2023-02-01 00:15:00,0,2,0,1.333333,1
9,82,2023-02-01 00:15:00,0,2,2,1.250000,1
